# Atividade 6 - desocupacao e afazeres domesticos

Nesta atividade juntamos as Tabelas 5 de 2012 e 2026, comparamos os percentuais por sexo e cruzamos com a Tabela 1.1.1 do IBGE sobre horas semanais dedicadas a cuidados e afazeres domesticos.

## 1. Importar bibliotecas

In [ ]:
import re
import unicodedata

import pandas as pd

## 2. Abrir as duas Tabelas 5

In [ ]:
t2012 = pd.read_csv("Tabela5-sem_emprego_2012.csv", sep=";", decimal=",")
t2026 = pd.read_csv("Tabela5-sem_emprego_2026.csv", sep=";", decimal=",")

display(t2012.head())
display(t2026.head())

## 3. Juntar 2012 e 2026

In [ ]:
comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")

comp["var_homens_pp"] = (
    comp["Desocupados - homens (2026 T1)"]
    - comp["Desocupados - homens (2012 T1)"]
)
comp["var_mulheres_pp"] = (
    comp["Desocupados - mulheres (2026 T1)"]
    - comp["Desocupados - mulheres (2012 T1)"]
)

comp.head()

## 4. Ler e limpar a Tabela 1.1.1 da Aula 2

In [ ]:
bruto = pd.read_excel("Tabela 1.1.1.xls", engine="xlrd", header=None)

# Linhas 0-7: titulo/cabecalho | 8-40: dados | 41+: fonte e notas
afazeres = bruto.iloc[8:41].copy()
afazeres.columns = [
    "Estado",
    "horas_total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
afazeres = afazeres.reset_index(drop=True)

for coluna in afazeres.columns[1:]:
    afazeres[coluna] = pd.to_numeric(afazeres[coluna], errors="coerce")

regioes = ["Brasil", "Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
afazeres_estados = afazeres[~afazeres["Estado"].isin(regioes)].copy()

afazeres_estados.head()

## 5. Cruzar desocupacao com afazeres domesticos

In [ ]:
def normalizar_nome(texto):
    texto = str(texto).strip().lower()
    texto = "".join(
        caractere
        for caractere in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^a-z0-9]+", " ", texto).strip()


comp["estado_chave"] = comp["Estado"].apply(normalizar_nome)
afazeres_estados["estado_chave"] = afazeres_estados["Estado"].apply(normalizar_nome)

cruzado = comp.merge(
    afazeres_estados,
    on="estado_chave",
    how="inner",
    suffixes=("", "_afazeres"),
)

cruzado.head()

## 6. Conferir resultado e salvar CSV

In [ ]:
print(f"Tabela 2012: {t2012.shape[0]} linhas")
print(f"Tabela 2026: {t2026.shape[0]} linhas")
print(f"Tabela cruzada: {cruzado.shape[0]} linhas")

faltantes = sorted(set(comp["Estado"]) - set(cruzado["Estado"]))
print("Estados sem cruzamento:", faltantes if faltantes else "nenhum")

cruzado.to_csv("geral.csv", sep=";", index=False, encoding="utf-8-sig")
cruzado.head()

## 7. Maiores variacoes

In [ ]:
maior_aumento_homens = cruzado.sort_values("var_homens_pp", ascending=False)[
    ["Sigla", "Estado", "var_homens_pp"]
].head(5)

maior_aumento_mulheres = cruzado.sort_values("var_mulheres_pp", ascending=False)[
    ["Sigla", "Estado", "var_mulheres_pp"]
].head(5)

display(maior_aumento_homens)
display(maior_aumento_mulheres)